The structure goes as follows.

Data load and extract metadata, both AIS and DAS. of course i cant upload DAS to GITHUB so an image will suffice. AIS is loaded as a zip and downloaded for a day.

Visualize data, cut the axis and prepare the images

afterward manual labeling

train CNN model on labeled data, using hp search grid search test on 20 samples.

Check performance metrics

align with AIS

calculate FFT to check validity of AIS


In [1]:
# Import required libraries
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta,timezone
import glob
import os
from scipy.fft import fft2, ifft2, fftshift, ifftshift
from scipy.signal import butter, filtfilt, firwin
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal.windows import tukey
from scipy.interpolate import interp1d
# from matplotlib.colors import LinearSegmentedColomap
from matplotlib.ticker import AutoMinorLocator

# Set up matplotlib for inline plotting
%matplotlib inline


In [2]:
def extract_metadata(file_path):
    """Extract metadata from HDF5 file"""
    with h5py.File(file_path, "r") as hdf5_file:
        start_time = hdf5_file["header/time"][()]
        dt = hdf5_file["header/dt"][()]
        dx = hdf5_file["header/dx"][()]
        channels = hdf5_file["header/channels"][()]
        # Get the actual number of samples in the file
        num_samples = hdf5_file["data"].shape[0]
    return start_time, dt, dx, channels, num_samples
# Get the first 6 files from the directory
directory = "raw_data/GC_data/20250702"
files = sorted(os.listdir(directory))[:6]

# Get all HDF5 files in the directory
file_paths = sorted(glob.glob(os.path.join(directory, '*.hdf5')))
print(len(file_paths))
if not file_paths:
    print(f"No HDF5 files found in {directory}")
else:
    # Extract metadata from the first file
    start_time, dt, dx, channels, num_samples = extract_metadata(file_paths[0])
    fs = 1/dt

    print("Metadata from first file:")
    print(f"Start time (Unix timestamp): {start_time}")
    print(f"Start time (UTC): {datetime.fromtimestamp(start_time, tz=timezone.utc)}")
    print(f"Sampling interval (dt): {dt} seconds")
    print(f"Spatial sampling (dx): {dx} meters")
    print(f"Number of channels: {len(channels)}")
    print(f"Samples per file: {num_samples}")
    print(f"Sampling frequency: {fs} Hz")
    print(f"File duration: {num_samples * dt} seconds")
    print(f"Total files available: {len(file_paths)}")
    print(f"Maximum duration available: {len(file_paths) * num_samples * dt} seconds")

341
Metadata from first file:
Start time (Unix timestamp): 1751458716.152
Start time (UTC): 2025-07-02 12:18:36.152000+00:00
Sampling interval (dt): 0.0016 seconds
Spatial sampling (dx): 1.0213001907746815 meters
Number of channels: 13752
Samples per file: 6250
Sampling frequency: 625.0 Hz
File duration: 10.0 seconds
Total files available: 341
Maximum duration available: 3410.0 seconds


In [ ]:
# Use existing variables from notebook
INPUT_DIR = directory
OUTPUT_DIR = "unseen_data"
FS_ASSUMED = fs
DX_ASSUMED_M = dx
DIST_MIN_KM = 4
DIST_MAX_KM = 14
VMAX = 850
IMG_H, IMG_W = 512, 1024
CMAP = "jet"
DPI = 150

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Use first 3 files from file_paths
if len(file_paths) < 3:
    raise ValueError("Need at least 3 files")

p1, p2, p3 = file_paths[0], file_paths[1], file_paths[2]
bases = [os.path.splitext(os.path.basename(p))[0] for p in (p1, p2, p3)]

try:
    slices = []
    for p in (p1, p2, p3):
        with h5py.File(p, 'r') as f:
            dset = f['strainrate'] if 'strainrate' in f else f['data']
            T, C = dset.shape
            distances_km = (np.arange(C) * DX_ASSUMED_M) / 1000.0
            ch_lo = int(math.ceil((DIST_MIN_KM * 1000.0) / DX_ASSUMED_M))
            ch_hi = int(math.floor((DIST_MAX_KM * 1000.0) / DX_ASSUMED_M)) + 1
            ch_lo, ch_hi = max(0, ch_lo), min(ch_hi, C)
            window = dset[:, ch_lo:ch_hi].astype(np.float32)
            slices.append(window)
    
    stitched = np.concatenate(slices, axis=0)
    stitched = np.clip(stitched, 0.0, VMAX) / VMAX
    
    fig = plt.figure(figsize=(IMG_W / DPI, IMG_H / DPI), dpi=DPI)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis('off')
    ax.imshow(stitched, aspect='auto', cmap=CMAP, vmin=0.0, vmax=1.0, interpolation='nearest')
    
    out_name = f"{bases[0]}_T30s.png"
    out_path = os.path.join(OUTPUT_DIR, out_name)
    plt.show()
    plt.close(fig)
    
    print(f"Saved {out_name}: shape {stitched.shape}")
except Exception as e:
    print(f"Error: {e}")


D